In [0]:
6#read csv using pyspark
df=spark.read \
  .format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/datafiles/nyc_taxi.csv")
df.display()

In [0]:
%sql
select count(*) from workspace.default.nyc_taxi

In [0]:
#pandas vs pyspark

import pandas as pd
pdf=pd.read_csv("/Volumes/workspace/default/datafiles/nyc_taxi.csv")
pdf.head()

#pyspark (distributed)
sdf=spark.read.option("header","true").csv("/Volumes/workspace/default/datafiles/nyc_taxi.csv")

#pandas runs on driver memory, pyspark is distributed

In [0]:
#read csv using pyspark
df=spark.read \
  .format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/datafiles/taxi_tripdata.csv")
#spark SQL
df.createOrReplaceTempView("taxi")
spark.sql("""SELECT passenger_count, count(*) as trips
          from taxi
          group by passenger_count
          order by trips DESC""").display()

In [0]:
#write delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/datafiles/mini_taxi_delta")

spark.read.format("delta").load("/Volumes/workspace/default/datafiles/mini_taxi_delta").display()

In [0]:
# MERGE (UPSERT) - simulate incremental data
from delta.tables import DeltaTable

delta_table=DeltaTable.forPath(spark, "/Volumes/workspace/default/datafiles/silver_taxi")
updates_df=spark.read.format("delta") \
    .load("/Volumes/workspace/default/datafiles/silver_taxi").limit(100)

updates_df = updates_df.dropDuplicates(["lpep_pickup_datetime"])

delta_table.alias("t").merge(
    updates_df.alias("u"),
    "t.lpep_pickup_datetime=u.lpep_pickup_datetime") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
%sql
select * from delta.`/Volumes/workspace/default/datafiles/silver_taxi`

In [0]:
# shuffle partitioning
silver_df=spark.read.format("delta").load("/Volumes/workspace/default/datafiles/silver_taxi")
spark.conf.set("spark.sql.shuffle.partitions","8")
# silver_df.cache()  do not use .cache() or .persist() on serverless clusters
silver_df.count()

In [0]:
# file based streaming
raw_df = spark.read \
    .option("header", "true") \
    .option("inferschema", "true") \
    .csv("/Volumes/workspace/default/datafiles/taxi_tripdata.csv")
stream_df=spark.readStream.schema(raw_df.schema) \
    .option("maxFilesPerTrigger",1) \
    .csv("/Volumes/workspace/default/datafiles/streaming_input")

stream_df.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/datafiles/taxi_streaming_checkpoint") \
    .start("/Volumes/workspace/default/datafiles/taxi_streaming_output")